# Pre-Exam Practice --- SOLUTIONS

**Surface Phenomena & Catalysis (2302337)**

This notebook contains complete, worked solutions for every problem in the
practice exercise. Run all cells (`Kernel > Restart & Run All`) to verify.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# Constants
R = 8.314              # J/(mol*K)
k_B = 1.380649e-23     # J/K  (Boltzmann constant)
h = 6.62607015e-34     # J*s  (Planck constant)
N_A = 6.022e23         # mol^-1
sigma_N2 = 0.162e-18   # m^2 (N2 cross-section at 77 K)
V_mol_STP = 22414      # cm^3/mol at STP

# Colorblind-safe palette (Wong 2011)
CB_BLUE = '#0072B2'
CB_ORANGE = '#E69F00'
CB_GREEN = '#009E73'
CB_VERMILLION = '#D55E00'
CB_SKYBLUE = '#56B4E9'
CB_PURPLE = '#CC79A7'

plt.rcParams.update({'font.size': 12, 'figure.figsize': (8, 5)})

---
## Problem 1: BET Analysis --- Solutions

### (a) Load data, BET transform, and plot

In [ ]:
# Load BET data
df_bet = pd.read_csv('data/practice_bet.csv')
print("Raw data:")
print(df_bet.to_string(index=False))

# Select BET range: 0.05 <= P/P0 <= 0.30
mask = (df_bet['P_over_P0'] >= 0.05) & (df_bet['P_over_P0'] <= 0.30)
df_bet_range = df_bet[mask].copy()

# BET transform: y = (P/P0) / [V_ads * (1 - P/P0)]
x = df_bet_range['P_over_P0'].values
V = df_bet_range['V_ads_cm3_STP_per_g'].values
y_bet = x / (V * (1 - x))

# Plot
fig, ax = plt.subplots()
ax.plot(x, y_bet, 'o', color=CB_BLUE, markersize=8, label='BET data')
ax.set_xlabel(r'$P/P_0$')
ax.set_ylabel(r'$(P/P_0)\,/\,[V_{\mathrm{ads}}\,(1 - P/P_0)]$'
              '  (g/cm$^3$ STP)')
ax.set_title('BET Transform')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (b) Linear regression: $V_m$ and $c$

In [ ]:
# Linear regression on BET transform
slope, intercept, r_val, p_val, se = stats.linregress(x, y_bet)

# Extract V_m and c
V_m = 1.0 / (slope + intercept)
c = slope / intercept + 1.0

print(f"Slope     = {slope:.6f} g/(cm^3 STP)")
print(f"Intercept = {intercept:.6f} g/(cm^3 STP)")
print(f"R^2       = {r_val**2:.6f}")
print(f"V_m       = {V_m:.2f} cm^3(STP)/g")
print(f"c         = {c:.1f}")

# Overlay regression line on BET plot
fig, ax = plt.subplots()
ax.plot(x, y_bet, 'o', color=CB_BLUE, markersize=8, label='BET data')
x_fit = np.linspace(x.min(), x.max(), 100)
ax.plot(x_fit, slope * x_fit + intercept, '-', color=CB_ORANGE,
        linewidth=2, label=f'Fit ($R^2$ = {r_val**2:.5f})')
ax.set_xlabel(r'$P/P_0$')
ax.set_ylabel(r'$(P/P_0)\,/\,[V_{\mathrm{ads}}\,(1 - P/P_0)]$'
              '  (g/cm$^3$ STP)')
ax.set_title('BET Linear Regression')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (c) BET surface area

In [ ]:
# S_BET = V_m * N_A * sigma_N2 / V_mol_STP
S_BET = V_m * N_A * sigma_N2 / V_mol_STP

print(f"S_BET = {S_BET:.1f} m^2/g")
print(f"Expected: ~348 m^2/g")
print(f"Deviation: {abs(S_BET - 348)/348*100:.1f}%")

# Alternative using shortcut: S = V_m * 4.35
S_shortcut = V_m * 4.3525
print(f"\nShortcut check: V_m * 4.35 = {S_shortcut:.1f} m^2/g")

### (d) Interpretation of $c$

\
**Model answer:**

The BET constant $c \approx 150$ indicates a strong interaction between N$_2$
and the $\gamma$-Al$_2$O$_3$ surface. Specifically, $c \approx \exp[(E_1 - E_L)/(RT)]$,
so a large $c$ means the heat of adsorption for the first monolayer ($E_1$)
significantly exceeds the heat of liquefaction of N$_2$ ($E_L$). This results
in a sharp "knee" in the isotherm at low $P/P_0$, where the monolayer fills
nearly completely before multilayer adsorption begins.

Values of $c > 100$ are typical for oxide surfaces with polar hydroxyl groups
that interact strongly with the N$_2$ quadrupole.

---
## Problem 2: Arrhenius / Eyring Analysis --- Solutions

### (a) Load data and display table

In [ ]:
# Load Arrhenius data
df_arr = pd.read_csv('data/practice_arrhenius.csv')

T = df_arr['T_K'].values.astype(float)
k_data = df_arr['k_per_s'].values

# Derived quantities
df_arr['1/T (K^-1)'] = 1.0 / T
df_arr['ln(k)'] = np.log(k_data)
df_arr['ln(k/T)'] = np.log(k_data / T)

print(df_arr.to_string(index=False, float_format='%.4f'))

### (b) Arrhenius plot

In [ ]:
# Arrhenius regression: ln(k) vs 1/T
inv_T = 1.0 / T
ln_k = np.log(k_data)

slope_arr, intercept_arr, r_arr, _, se_arr = stats.linregress(inv_T, ln_k)
Ea = -slope_arr * R             # J/mol
A_pre = np.exp(intercept_arr)   # s^-1

print("Arrhenius Results:")
print(f"  Slope     = {slope_arr:.1f} K")
print(f"  E_a       = {Ea/1000:.2f} kJ/mol")
print(f"  A         = {A_pre:.3e} s^-1")
print(f"  R^2       = {r_arr**2:.6f}")

# Plot
fig, ax = plt.subplots()
ax.plot(inv_T * 1000, ln_k, 'o', color=CB_BLUE, markersize=8,
        label='Data')
inv_T_fit = np.linspace(inv_T.min(), inv_T.max(), 100)
ax.plot(inv_T_fit * 1000, slope_arr * inv_T_fit + intercept_arr, '-',
        color=CB_ORANGE, linewidth=2,
        label=f'Fit ($R^2$ = {r_arr**2:.5f})')
ax.set_xlabel(r'$1000/T$ (K$^{-1}$)')
ax.set_ylabel(r'$\ln\,k$')
ax.set_title('Arrhenius Plot')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (c) Eyring plot

In [ ]:
# Eyring regression: ln(k/T) vs 1/T
ln_k_over_T = np.log(k_data / T)

slope_eyr, intercept_eyr, r_eyr, _, _ = stats.linregress(inv_T,
                                                           ln_k_over_T)

dH_ddagger = -slope_eyr * R                            # J/mol
dS_ddagger = (intercept_eyr - np.log(k_B / h)) * R     # J/(mol*K)

print("Eyring Results:")
print(f"  Slope        = {slope_eyr:.1f} K")
print(f"  dH_ddagger   = {dH_ddagger/1000:.2f} kJ/mol")
print(f"  dS_ddagger   = {dS_ddagger:.2f} J/(mol*K)")
print(f"  R^2          = {r_eyr**2:.6f}")

# Plot
fig, ax = plt.subplots()
ax.plot(inv_T * 1000, ln_k_over_T, 's', color=CB_GREEN, markersize=8,
        label='Data')
ax.plot(inv_T_fit * 1000,
        slope_eyr * inv_T_fit + intercept_eyr, '-',
        color=CB_VERMILLION, linewidth=2,
        label=f'Fit ($R^2$ = {r_eyr**2:.5f})')
ax.set_xlabel(r'$1000/T$ (K$^{-1}$)')
ax.set_ylabel(r'$\ln(k/T)$')
ax.set_title('Eyring Plot')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (d) Thermodynamic consistency check

In [ ]:
# Verify: Ea ~ dH_ddagger + R * T_mean
T_mean = np.mean(T)
Ea_predicted = dH_ddagger + R * T_mean  # J/mol

discrepancy = abs(Ea - Ea_predicted) / Ea * 100

print(f"E_a (Arrhenius)            = {Ea/1000:.2f} kJ/mol")
print(f"dH_ddagger + R*T_mean      = {Ea_predicted/1000:.2f} kJ/mol")
print(f"  (dH = {dH_ddagger/1000:.2f} + R*{T_mean:.0f}"
      f" = {R*T_mean/1000:.2f} kJ/mol)")
print(f"Discrepancy                = {discrepancy:.2f}%")
print()
print("This confirms E_a ~ dH_ddagger + R*T_mean (within noise).")

---
## Problem 3: Transport Limitations --- Solutions

### (a) $\phi$, $\eta$, $r_\text{obs}$

In [ ]:
# Given parameters
k_tp = 5.0           # s^-1
D_eff = 1.0e-6       # m^2/s
R_p = 2.0e-3         # m  (2.0 mm -> m)
C_As = 10.0          # mol/m^3
Ea_tp = 85.0         # kJ/mol

# Thiele modulus (sphere)
phi = R_p * np.sqrt(k_tp / D_eff)
print(f"Thiele modulus: phi = {phi:.4f}")

# Effectiveness factor (sphere formula)
coth_phi = np.cosh(phi) / np.sinh(phi)
eta = (3.0 / phi**2) * (phi * coth_phi - 1.0)
print(f"Effectiveness factor: eta = {eta:.4f}")

# Observed rate
r_obs = eta * k_tp * C_As
print(f"Observed rate: r_obs = {r_obs:.2f} mol/(m^3*s)")

# Regime identification
if phi < 0.3:
    regime = "Kinetic-controlled (phi < 0.3)"
elif phi < 3.0:
    regime = "Transition regime (0.3 < phi < 3)"
else:
    regime = "Diffusion-limited (phi > 3)"
print(f"Regime: {regime}")

### (b) $\eta$ vs $\phi$ log-log plot

In [ ]:
# Effectiveness factor curve
phi_range = np.logspace(-2, 2, 200)
coth_range = np.cosh(phi_range) / np.sinh(phi_range)
eta_range = (3.0 / phi_range**2) * (phi_range * coth_range - 1.0)

# Asymptote: eta = 3/phi (diffusion-limited)
eta_asymp = 3.0 / phi_range

fig, ax = plt.subplots()
ax.loglog(phi_range, eta_range, '-', color=CB_BLUE, linewidth=2.5,
          label=r'$\eta(\phi)$ (sphere)')
ax.loglog(phi_range, eta_asymp, '--', color='gray', linewidth=1.5,
          label=r'Asymptote: $\eta = 3/\phi$')

# Mark operating point
ax.plot(phi, eta, 'o', color=CB_VERMILLION, markersize=12, zorder=5,
        label=f'Operating point')
ax.annotate(f'  phi={phi:.2f}, eta={eta:.3f}',
            xy=(phi, eta), fontsize=10, color=CB_VERMILLION)

ax.set_xlabel(r'Thiele modulus $\phi$')
ax.set_ylabel(r'Effectiveness factor $\eta$')
ax.set_title('Effectiveness Factor vs Thiele Modulus (Sphere)')
ax.set_xlim(0.01, 100)
ax.set_ylim(0.01, 1.5)
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### (c) Weisz--Prater diagnosis

In [ ]:
# Weisz-Prater number
N_WP = r_obs * R_p**2 / (D_eff * C_As)

print(f"Weisz-Prater number: N_WP = {N_WP:.2f}")
print(f"  (Equivalent check: eta * phi^2 = {eta * phi**2:.2f})")

if N_WP < 0.3:
    print("Diagnosis: N_WP << 1 -- internal diffusion is NOT significant.")
else:
    print("Diagnosis: N_WP >> 1 -- SEVERE internal diffusion limitation.")

### (d) Pellet size sweep

In [ ]:
# Pellet size sweep
R_p_sweep = np.linspace(0.1e-3, 5.0e-3, 100)  # m
phi_sweep = R_p_sweep * np.sqrt(k_tp / D_eff)
coth_sweep = np.cosh(phi_sweep) / np.sinh(phi_sweep)
eta_sweep = (3.0 / phi_sweep**2) * (phi_sweep * coth_sweep - 1.0)

# Find R_p_crit where eta = 0.95
# eta is monotonically decreasing with R_p, so reverse for np.interp
R_p_sweep_mm = R_p_sweep * 1000  # mm
R_p_crit_mm = np.interp(0.95, eta_sweep[::-1], R_p_sweep_mm[::-1])

print(f"Critical pellet radius: R_p,crit = {R_p_crit_mm:.2f} mm"
      f"  (where eta = 0.95)")

fig, ax = plt.subplots()
ax.plot(R_p_sweep_mm, eta_sweep, '-', color=CB_BLUE, linewidth=2.5)
ax.axhline(0.95, color='gray', linestyle='--', linewidth=1,
           label=r'$\eta = 0.95$')
ax.axvline(R_p_crit_mm, color=CB_VERMILLION, linestyle='--', linewidth=1,
           label=f'$R_{{p,crit}}$ = {R_p_crit_mm:.2f} mm')
ax.plot(R_p * 1000, eta, 'o', color=CB_VERMILLION, markersize=10,
        label=f'Operating point ({R_p*1000:.1f} mm)')
ax.set_xlabel(r'Pellet radius $R_p$ (mm)')
ax.set_ylabel(r'Effectiveness factor $\eta$')
ax.set_title('Effect of Pellet Size on Effectiveness Factor')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (e) Apparent activation energy

In [ ]:
Ea_app = Ea_tp / 2.0
print(f"Intrinsic E_a = {Ea_tp:.1f} kJ/mol")
print(f"Apparent E_a  = {Ea_app:.1f} kJ/mol  (under strong diffusion "
      f"limitation)")

\
**Explanation:**

Under strong diffusion limitation ($\phi \gg 1$), the reactant is consumed
before it can penetrate to the pellet centre. The observed rate is controlled
by both the intrinsic kinetics *and* the diffusion rate. Because $D_\text{eff}$
has a much weaker temperature dependence ($\sim T^{0.5\text{--}1.75}$) compared
to the exponential Arrhenius dependence of $k$, the overall apparent activation
energy becomes:

$$E_a^{\text{app}} = \frac{E_a^{\text{intrinsic}} + E_{D}}{2}
\approx \frac{E_a^{\text{intrinsic}}}{2}$$

This "halving" arises because the effectiveness factor $\eta \propto 1/\phi
\propto 1/\sqrt{k}$, so $r_\text{obs} = \eta \cdot k \cdot C_{A,s}
\propto \sqrt{k} \propto \exp(-E_a/(2RT))$.

---
## Problem 4: Selectivity Engineering --- Solutions

### (a) $\tau^*_{\mathrm{PFR}}$ and $Y_{B,\max}$

In [ ]:
# Rate constants at T_ref = 450 K
k1 = 0.80  # s^-1
k2 = 0.50  # s^-1
C_A0 = 1.0  # mol/L

# Optimal PFR space time
tau_star = np.log(k2 / k1) / (k2 - k1)
print(f"tau*_PFR = {tau_star:.4f} s")

# Maximum yield of B
Y_B_max = (k1 / (k2 - k1)) * (np.exp(-k1 * tau_star)
                                - np.exp(-k2 * tau_star))
print(f"Y_B_max  = {Y_B_max:.4f}")

### (b) Concentration profiles

In [ ]:
# Concentration profiles in PFR
tau = np.linspace(0, 10, 300)
C_A = C_A0 * np.exp(-k1 * tau)
C_B = C_A0 * (k1 / (k2 - k1)) * (np.exp(-k1 * tau) - np.exp(-k2 * tau))
C_C = C_A0 - C_A - C_B

fig, ax = plt.subplots()
ax.plot(tau, C_A, '-', color=CB_BLUE, linewidth=2, label=r'$C_A$')
ax.plot(tau, C_B, '-', color=CB_ORANGE, linewidth=2, label=r'$C_B$')
ax.plot(tau, C_C, '-', color=CB_GREEN, linewidth=2, label=r'$C_C$')
ax.axvline(tau_star, color='gray', linestyle='--', linewidth=1.5,
           label=f'tau* = {tau_star:.2f} s')

# Mark Y_B_max
ax.plot(tau_star, Y_B_max * C_A0, 'o', color=CB_ORANGE, markersize=10,
        zorder=5)
ax.annotate(f'  Y_B_max = {Y_B_max:.3f}',
            xy=(tau_star, Y_B_max * C_A0), fontsize=10,
            color=CB_ORANGE)

ax.set_xlabel(r'Space time $\tau$ (s)')
ax.set_ylabel('Concentration (mol/L)')
ax.set_title(r'PFR Concentration Profiles: A $\to$ B $\to$ C')
ax.legend(loc='right')
ax.set_xlim(0, 10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### (c) CSTR comparison

In [ ]:
# CSTR optimal space time and yield
tau_cstr = 1.0 / np.sqrt(k1 * k2)
Y_cstr = (k1 * tau_cstr
          / ((1 + k1 * tau_cstr) * (1 + k2 * tau_cstr)))

# PFR advantage
advantage = (Y_B_max - Y_cstr) / Y_cstr * 100

print("CSTR:")
print(f"  tau*_CSTR     = {tau_cstr:.4f} s")
print(f"  Y_B_max_CSTR  = {Y_cstr:.4f}")
print()
print("PFR:")
print(f"  tau*_PFR      = {tau_star:.4f} s")
print(f"  Y_B_max_PFR   = {Y_B_max:.4f}")
print()
print(f"PFR yield advantage over CSTR: {advantage:.1f}%")
print()
print("The PFR always gives higher intermediate yield because it "
      "avoids the")
print("back-mixing that prematurely converts B -> C in a CSTR.")

### (d) Crossover temperature

In [ ]:
# Arrhenius parameters (self-consistent with k at T_ref)
A1 = 1.0685e+08   # s^-1
A2 = 1.2118e+06   # s^-1
Ea1 = 70e3   # J/mol
Ea2 = 55e3   # J/mol

# Crossover temperature
T_cross = (Ea1 - Ea2) / (R * np.log(A1 / A2))
print(f"T_cross = {T_cross:.1f} K")
print()

# Verify: at T_cross, k1 should equal k2
k1_cross = A1 * np.exp(-Ea1 / (R * T_cross))
k2_cross = A2 * np.exp(-Ea2 / (R * T_cross))
print("Verification at T_cross:")
print(f"  k1(T_cross) = {k1_cross:.4f} s^-1")
print(f"  k2(T_cross) = {k2_cross:.4f} s^-1")

\
**Strategy:**

Since $E_a^{(1)} > E_a^{(2)}$, $k_1$ increases faster with temperature than
$k_2$. Therefore, **raising $T$ above $T_\text{cross}$** increases the ratio
$k_1/k_2 > 1$, which means A is consumed faster relative to B. This
**improves selectivity** toward B (larger $Y_{B,\max}$).

Recommendation: operate above $T_\text{cross}$ to maximise B yield, but not
so high that thermal degradation or equilibrium limitations become significant.

### (e) Rate constants at multiple temperatures

In [ ]:
# Multi-temperature analysis
T_test = np.array([400, 450, 500], dtype=float)

header = (f"{'T (K)':<8} {'k1 (s^-1)':<12} {'k2 (s^-1)':<12} "
          f"{'k1/k2':<8} {'tau* (s)':<10} {'Y_B_max':<8}")
print(header)
print("-" * len(header))

for T_val in T_test:
    k1_T = A1 * np.exp(-Ea1 / (R * T_val))
    k2_T = A2 * np.exp(-Ea2 / (R * T_val))
    ratio = k1_T / k2_T

    # Optimal space time and yield
    tau_opt = np.log(k2_T / k1_T) / (k2_T - k1_T)
    Y_opt = (k1_T / (k2_T - k1_T)) * (np.exp(-k1_T * tau_opt)
                                        - np.exp(-k2_T * tau_opt))

    print(f"{T_val:<8.0f} {k1_T:<12.4f} {k2_T:<12.4f} "
          f"{ratio:<8.3f} {tau_opt:<10.4f} {Y_opt:<8.4f}")

print()
print("Trend: As T increases above T_cross, k1/k2 increases,")
print("tau* decreases (faster reaction), and Y_B_max increases")
print("(better selectivity toward B).")

---
## Summary of Answers

In [ ]:
print("=" * 65)
print("PRACTICE EXAM --- SUMMARY OF KEY ANSWERS")
print("=" * 65)

print("\nProblem 1: BET Analysis")
print(f"  V_m     = {V_m:.2f} cm^3(STP)/g  (expected ~80)")
print(f"  c       = {c:.1f}                 (expected ~150)")
print(f"  S_BET   = {S_BET:.1f} m^2/g       (expected ~348)")

print("\nProblem 2: Arrhenius / Eyring")
print(f"  E_a        = {Ea/1000:.2f} kJ/mol     (expected ~79)")
print(f"  dH_ddagger = {dH_ddagger/1000:.2f} kJ/mol  (expected 75)")
print(f"  dS_ddagger = {dS_ddagger:.2f} J/(mol*K) (expected -35)")

print("\nProblem 3: Transport Limitations")
print(f"  phi       = {phi:.4f}             (expected ~4.47)")
print(f"  eta       = {eta:.4f}             (expected ~0.521)")
print(f"  r_obs     = {r_obs:.2f} mol/(m^3*s)")
print(f"  N_WP      = {N_WP:.2f}              (expected ~10.4)")
print(f"  R_p,crit  = {R_p_crit_mm:.2f} mm          (expected ~0.40)")
print(f"  E_a,app   = {Ea_app:.1f} kJ/mol     (expected 42.5)")

print("\nProblem 4: Selectivity Engineering")
print(f"  tau*_PFR     = {tau_star:.4f} s      (expected ~1.57)")
print(f"  Y_B,max_PFR  = {Y_B_max:.4f}        (expected ~0.457)")
print(f"  tau*_CSTR    = {tau_cstr:.4f} s      (expected ~1.58)")
print(f"  Y_B,max_CSTR = {Y_cstr:.4f}        (expected ~0.312)")
print(f"  PFR advantage = {advantage:.1f}%")
print(f"  T_cross      = {T_cross:.1f} K       (expected ~403)")

print("\n" + "=" * 65)
print("All values should be close to expectations (within noise).")
print("Small deviations are due to the 1% Gaussian noise in the data.")
print("=" * 65)